# 02 — Question Segmentation

**Input:** `data/processed/raw_english_pages.json`

**Output:** `data/processed/questions.json`

**What this does:**
- Skips cover page and general instructions
- Merges pages per paper into one text block
- Splits by question number (1–38)
- Handles OR questions — stores both parts
- Handles Case Study sub-questions (i), (ii), (iii)
- Tags each question with section, marks, question type

---

## 1. Setup

In [1]:
import sys
sys.path.append('..')

import re
import json
from pathlib import Path
from collections import defaultdict

PROCESSED_DIR = Path('../data/processed')

with open(PROCESSED_DIR / 'raw_english_pages.json') as f:
    pages = json.load(f)

print(f'Loaded {len(pages)} pages')

Loaded 486 pages


## 2. Filter out cover and instructions pages

In [2]:
SKIP_PATTERNS = [
    r'Series\s*:',
    r'General Instructions',
    r'Roll No\.?',
]

def is_skip_page(text):
    return any(re.search(p, text) for p in SKIP_PATTERNS)

question_pages = [p for p in pages if not is_skip_page(p['text'])]
print(f'Question pages: {len(question_pages)}')
for p in question_pages:
    print(f"  Page {p['page_num']}: {p['text'][:60].strip()}")

Question pages: 431
  Page 4: 2. (tan A cosec A)2 (sin A sec A)2
(A) 0
(B) 1
(C) 1
(D) 2
3
  Page 5: 2. The value of (tan A cosec A)2 (sin A sec A)2 is :
(A) 0
(
  Page 7: 6. Two polynomials are shown in the graph below. The number
  Page 9: 9. If HCF(98, 28) = m and LCM(98, 28) = n, then the value of
  Page 11: 13. If a sector of a circle has an area of 40 sq. units and
  Page 13: 18. A card is drawn at random from a pack of 52 cards. What
  Page 15: 20. Assertion (A) : If we join two hemispheres of same radiu
  Page 16: 23. 26 m P
A B 10 m PA PB
-
24. p(x) = x2 + x
25. ABC, A(9,
  Page 17: 23. A person is standing at P outside a circular ground at a
  Page 19: 29. A room is in the form of a cylinder surmounted by a hemi
  Page 21: (b) A train travels a distance of 480 km at a uniform speed.
  Page 23: SECTION E
This section has 3 case study based questions carr
  Page 25: (iii) (a) Find the height CE of the lighthouse [Use = 1·73]
  Page 27: Case Study 3
38. A brooch is a decorative 

## 3. Merge pages per paper into one text block

In [3]:
FOOTER_RE = re.compile(r'30/1/[\d/]+\s*#\s*\d+\|\s*P\s*a\s*g\s*e.*', re.IGNORECASE)

def clean_page(text):
    text = FOOTER_RE.sub('', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

papers = defaultdict(list)
for p in question_pages:
    papers[p['paper']].append(p)

paper_texts = {}
for paper_name, paper_pages in papers.items():
    sorted_pages = sorted(paper_pages, key=lambda x: x['page_num'])
    merged = '\n\n'.join(clean_page(p['text']) for p in sorted_pages)
    paper_texts[paper_name] = {
        'year':  sorted_pages[0]['year'],
        'paper': paper_name,
        'text':  merged
    }
    print(f'{paper_name}: merged {len(merged)} chars')

print()
print('--- Merged text preview ---')
sample = list(paper_texts.values())[0]
print(sample['text'][:800])

Mathematics_Standard_30_1_3: merged 8868 chars
Mathematics_Standard_30_S_1_Supplementary_v2: merged 11574 chars
Mathematics_Standard_30_2_3: merged 9434 chars
Mathematics_Standard_30_B_S_Blind_Supplementary: merged 9063 chars
Mathematics_Basic_430_S_2_Supplementary: merged 9905 chars
Mathematics_Basic_430_1_1: merged 9683 chars
Mathematics_Basic_430_S_2_Supplementary_v2: merged 9075 chars
Mathematics_Standard_30_2_2: merged 10044 chars
Mathematics_Basic_430_3_3: merged 9133 chars
Mathematics_Standard_30_S_2_Supplementary_v2: merged 11624 chars
Mathematics_Standard_30_S_3_Supplementary: merged 11013 chars
Mathematics_Basic_430_B_S_Blind_Supplementary: merged 8548 chars
Mathematics_Basic_430_2_2: merged 8980 chars
Mathematics_Basic_430_2_3: merged 8626 chars
Mathematics_Standard_30_3_1: merged 9480 chars
Mathematics_Basic_430_3_2: merged 9321 chars
Mathematics_Standard_30_S_3_Supplementary_v2: merged 11633 chars
Mathematics_Standard_30_3_3: merged 9560 chars
Mathematics_Basic_430_S_3_Sup

## 4. Section and marks lookup

In [4]:
SECTION_MARKS = {'A': 1, 'B': 2, 'C': 3, 'D': 5, 'E': 4}
SECTION_TYPES = {'A': 'MCQ', 'B': 'VSA', 'C': 'SA', 'D': 'LA', 'E': 'CaseStudy'}
SECTION_Q_RANGES = {
    'A': range(1, 21),
    'B': range(21, 26),
    'C': range(26, 32),
    'D': range(32, 36),
    'E': range(36, 39)
}

def get_section(q_num):
    for section, r in SECTION_Q_RANGES.items():
        if q_num in r:
            return section
    return 'Unknown'

# Sanity check
for n in [1, 19, 21, 26, 32, 36, 38]:
    s = get_section(n)
    print(f'Q{n} -> Section {s} | {SECTION_TYPES[s]} | {SECTION_MARKS[s]} marks')

Q1 -> Section A | MCQ | 1 marks
Q19 -> Section A | MCQ | 1 marks
Q21 -> Section B | VSA | 2 marks
Q26 -> Section C | SA | 3 marks
Q32 -> Section D | LA | 5 marks
Q36 -> Section E | CaseStudy | 4 marks
Q38 -> Section E | CaseStudy | 4 marks


## 5. Split merged text into question blocks

In [5]:
Q_START = re.compile(r'^(\d{1,2})[\.\)]\s+', re.MULTILINE)

def split_into_blocks(text):
    matches = list(Q_START.finditer(text))
    blocks = []
    for i, match in enumerate(matches):
        q_num = int(match.group(1))
        if q_num < 1 or q_num > 38:
            continue
        start = match.start()
        end   = matches[i+1].start() if i+1 < len(matches) else len(text)
        blocks.append((q_num, text[start:end].strip()))
    return blocks

sample_text = list(paper_texts.values())[0]['text']
blocks = split_into_blocks(sample_text)
print(f'Found {len(blocks)} blocks')
for q_num, block in blocks[:4]:
    print(f'\n--- Q{q_num} ---')
    print(block[:200])

Found 47 blocks

--- Q2 ---
2. (tan A cosec A)2 (sin A sec A)2
(A) 0
(B) 1
(C) 1
(D) 2

--- Q3 ---
3. 150 m
30 :
(A) 100 m
(B) 300 m
(C) 150 m
(D) 150 m

--- Q4 ---
4. ABC DEF , B = E, F = C AB = 3DE ,
(A)
(B)
(C)
(D)

--- Q5 ---
5. 7 + 4 sin = 9 , :
(A) 90
(B) 30
(C) 45
(D) 60


## 6. OR question handler

In [6]:
OR_SPLIT = re.compile(r'\n\s*OR\s*\n', re.IGNORECASE)

def parse_or(block):
    parts = OR_SPLIT.split(block, maxsplit=1)
    if len(parts) == 2:
        return parts[0].strip(), parts[1].strip()
    return block.strip(), None

# Find and test an OR question
for q_num, block in blocks:
    if 'OR' in block:
        print(f'Testing Q{q_num} which has OR')
        a, b = parse_or(block)
        print(f'Part A: {a[:100]}')
        print(f'Part B: {b[:100]}')
        break

Testing Q21 which has OR
Part A: 21. (a) If ABC PQR in which AB = 6 cm, BC = 4 cm, AC = 8 cm and
PR = 6 cm, then find the length of (
Part B: (b) In the given figure, = and 1 = 2, show that
PQS TQR.


## 7. Case Study sub-question handler

In [7]:
SUB_Q = re.compile(r'\((i{1,3}|iv|v)\)\s+', re.IGNORECASE)

def parse_case_study_subs(q_num, block):
    sub_matches = list(SUB_Q.finditer(block))
    if not sub_matches:
        return [(str(q_num), block)]
    subs = []
    for i, m in enumerate(sub_matches):
        label = f"{q_num}({m.group(1)})"
        start = m.start()
        end   = sub_matches[i+1].start() if i+1 < len(sub_matches) else len(block)
        subs.append((label, block[start:end].strip()))
    return subs

# Test Q36
for q_num, block in blocks:
    if q_num == 36:
        subs = parse_case_study_subs(36, block)
        for label, t in subs:
            print(f'{label}: {t[:80]}')
        break

36(i): (i) 1
36(ii): (ii) 1
36(iii): (iii) (a) Find the height CE of the lighthouse [Use = 1·73] 2
OR
36(iii): (iii) (b) Find distance AE, if AC = 100 m. 2
Case Study 2


## 8. Clean question text

In [8]:
def clean_text(text):
    if not text:
        return ''
    text = re.sub(r'\([ABCD]\)\s+.+', '', text)  # remove MCQ options
    text = re.sub(r'30/1/\d+.*', '', text)         # remove footers
    text = re.sub(r'\n{2,}', ' ', text)            # collapse newlines
    text = re.sub(r'\s+', ' ', text)               # collapse spaces
    return text.strip()

# Test
raw = blocks[0][1]
print('Before:', raw[:200])
print()
print('After:', clean_text(raw)[:200])

Before: 2. (tan A cosec A)2 (sin A sec A)2
(A) 0
(B) 1
(C) 1
(D) 2

After: 2. (tan A cosec A)2 (sin A sec A)2


## 9. Build all question objects

In [9]:
def build_questions(paper_name, year, blocks):
    questions = []
    for q_num, block in blocks:
        section       = get_section(q_num)
        marks         = SECTION_MARKS.get(section, 0)
        question_type = SECTION_TYPES.get(section, 'Unknown')

        if section == 'E':
            # Case Study: split into sub-questions first
            subs = parse_case_study_subs(q_num, block)
            for label, sub_text in subs:
                part_a, part_b = parse_or(sub_text)
                questions.append({
                    'year':            year,
                    'paper':           paper_name,
                    'section':         section,
                    'question_type':   question_type,
                    'question_number': label,
                    'marks':           marks,
                    'text':            clean_text(part_a),
                    'has_or':          part_b is not None,
                    'or_text':         clean_text(part_b) if part_b else ''
                })
        else:
            part_a, part_b = parse_or(block)
            questions.append({
                'year':            year,
                'paper':           paper_name,
                'section':         section,
                'question_type':   question_type,
                'question_number': str(q_num),
                'marks':           marks,
                'text':            clean_text(part_a),
                'has_or':          part_b is not None,
                'or_text':         clean_text(part_b) if part_b else ''
            })
    return questions


all_questions = []
for paper_name, paper_data in paper_texts.items():
    blks = split_into_blocks(paper_data['text'])
    qs   = build_questions(paper_name, paper_data['year'], blks)
    all_questions.extend(qs)
    print(f'{paper_name}: {len(qs)} questions')

print(f'\nTotal: {len(all_questions)} questions')

Mathematics_Standard_30_1_3: 56 questions
Mathematics_Standard_30_S_1_Supplementary_v2: 46 questions
Mathematics_Standard_30_2_3: 46 questions
Mathematics_Standard_30_B_S_Blind_Supplementary: 46 questions
Mathematics_Basic_430_S_2_Supplementary: 40 questions
Mathematics_Basic_430_1_1: 59 questions
Mathematics_Basic_430_S_2_Supplementary_v2: 42 questions
Mathematics_Standard_30_2_2: 55 questions
Mathematics_Basic_430_3_3: 59 questions
Mathematics_Standard_30_S_2_Supplementary_v2: 45 questions
Mathematics_Standard_30_S_3_Supplementary: 42 questions
Mathematics_Basic_430_B_S_Blind_Supplementary: 43 questions
Mathematics_Basic_430_2_2: 44 questions
Mathematics_Basic_430_2_3: 45 questions
Mathematics_Standard_30_3_1: 58 questions
Mathematics_Basic_430_3_2: 60 questions
Mathematics_Standard_30_S_3_Supplementary_v2: 46 questions
Mathematics_Standard_30_3_3: 58 questions
Mathematics_Basic_430_S_3_Supplementary: 41 questions
Mathematics_Basic_430_3_1: 61 questions
Mathematics_Basic_430_B_S_Blin

## 10. Inspect results

In [10]:
import pandas as pd
df = pd.DataFrame(all_questions)

print('Questions per section:')
print(df['section'].value_counts().sort_index())
print()
print('OR questions:', df['has_or'].sum())
print()
print(df[['question_number','section','question_type','marks','has_or']].to_string())

Questions per section:
section
A    658
B    211
C    253
D    155
E    370
Name: count, dtype: int64

OR questions: 278

     question_number section question_type  marks  has_or
0                  2       A           MCQ      1   False
1                  3       A           MCQ      1   False
2                  4       A           MCQ      1   False
3                  5       A           MCQ      1   False
4                  2       A           MCQ      1   False
5                  3       A           MCQ      1   False
6                  4       A           MCQ      1   False
7                  5       A           MCQ      1   False
8                  6       A           MCQ      1   False
9                  7       A           MCQ      1   False
10                 8       A           MCQ      1   False
11                 9       A           MCQ      1   False
12                10       A           MCQ      1   False
13                11       A           MCQ      1   False
14      

In [11]:
# Spot check a few questions from each section
for section in ['A','B','C','D','E']:
    q = df[df['section'] == section].iloc[0].to_dict()
    print(f"--- Section {section} sample ---")
    print(f"  Q{q['question_number']} | {q['question_type']} | {q['marks']}m")
    print(f"  {q['text'][:120]}")
    print()

--- Section A sample ---
  Q2 | MCQ | 1m
  2. (tan A cosec A)2 (sin A sec A)2

--- Section B sample ---
  Q21 | VSA | 2m
  21. (a) If ABC PQR in which AB = 6 cm, BC = 4 cm, AC = 8 cm and PR = 6 cm, then find the length of (PQ + QR).

--- Section C sample ---
  Q26 | SA | 3m
  26.

--- Section D sample ---
  Q32 | LA | 5m
  32. (a) The perimeter of a right triangle is 60 cm and its hypotenuse is 25 cm. Find the lengths of other two sides of t

--- Section E sample ---
  Q36(i) | CaseStudy | 4m
  (i) 1



## 11. Save

In [12]:
out_path = PROCESSED_DIR / 'questions.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(all_questions, f, indent=2, ensure_ascii=False)

print(f'✅ Saved {len(all_questions)} questions → {out_path}')

✅ Saved 1647 questions → ../data/processed/questions.json


---
## ✅ Done
Output: `data/processed/questions.json`

Next: `03_classification.ipynb`

Once verified → copy to `src/stages/segment.py`